# `mk_lemma_processor` — Usage Notebook

Demonstrates the core logic behind `scripts/mk_lemma_processor.py`.

**What the script does**

Makura-kotoba (枕詞, pillow words) are fixed epithets in Old Japanese poetry.
They appear in the corpus as `MK` blocks whose component lines are annotated
with the sentinel ID `L099999`.

The script:
1. Finds every `L099999` occurrence and replaces it with a newly allocated real ID.
2. Creates a dictionary entry for each MK block with:
   - `.COMPOUND` back-references to its component words
   - `.MKTARGETNEW` references to the word immediately following the block
3. Optionally normalises existing MK entries that are missing `.COMPOUND` /
   `.MKTARGETNEW` lines (`NORMALISE_EXISTING = True`).

**Report files produced**
- `report_modified_lines.txt` — corpus lines where `L099999` was replaced
- `report_dict_modified.txt` — existing MK entries normalised
- `report_dict_added.txt` — new MK entries added

All cells read real data from `data/` — no files are modified.

---
## 0. Setup

In [ ]:
import sys
import os
import re
from collections import defaultdict

sys.path.insert(0, os.path.join('..', 'src'))

DICT_FILE = os.path.join('..', 'data', 'xml', 'dict', 'dictionary.xml')
TEXT_DIR  = os.path.join('..', 'data', 'xml', 'text')

print("Python:", sys.version.split()[0])


---
## 1. Finding L099999 Sentinels in the Corpus

Every line inside a makura-kotoba block carries `L099999` as a placeholder ID.
The script scans all corpus files to collect these blocks.

`L099997` is also reserved and is explicitly skipped during processing.

In [ ]:
from coj.core.corpus import CorpusDocument

TARGET_ID = "L099999"

sentinel_lines = []   # (filename, corpus_line_obj)

for fname in sorted(os.listdir(TEXT_DIR)):
    if not fname.endswith('.xml'):
        continue
    doc = CorpusDocument.from_file(os.path.join(TEXT_DIR, fname))
    for cl in doc.all_corpus_lines():
        if TARGET_ID in cl.fields:
            sentinel_lines.append((fname, cl))

print(f"L099999 sentinel lines found: {len(sentinel_lines)}")
print("\nFirst 5 examples:")
for fname, cl in sentinel_lines[:5]:
    print(f"  {fname}  {cl.to_text()[:90]}")


---
## 2. Identifying MK Blocks and Their Components

A makura-kotoba block is a contiguous group of corpus lines that all carry `L099999`
and share the same `MK` node in their syntactic path.

The script groups consecutive sentinel lines that belong to the same block, then
extracts:
- **Component words** (the phonemic forms of each line in the block)
- **Component lemma IDs** (from the lines immediately preceding the MK block, which
  are the individual-word annotations)

In [ ]:
def mk_prefix(fields):
    """Return the path up-to-and-including MK, or None."""
    if 'MK' not in fields:
        return None
    idx = fields.index('MK')
    return tuple(fields[:idx+1])

# Group sentinel lines by file, then by consecutive MK-prefix
blocks = []  # list of {'file': ..., 'lines': [corpus_line_obj, ...]}

by_file = defaultdict(list)
for fname, cl in sentinel_lines:
    by_file[fname].append(cl)

for fname, cls in by_file.items():
    current_block = None
    current_prefix = None
    for cl in cls:
        pfx = mk_prefix(cl.fields)
        if pfx == current_prefix and current_block:
            current_block['lines'].append(cl)
        else:
            if current_block:
                blocks.append(current_block)
            current_block  = {'file': fname, 'prefix': pfx, 'lines': [cl]}
            current_prefix = pfx
    if current_block:
        blocks.append(current_block)

print(f"MK blocks identified: {len(blocks)}")

# Show first block
b = blocks[0]
print(f"\nBlock 0 in {b['file']}:")
print(f"  MK path: {','.join(b['prefix'])}")
print(f"  Lines ({len(b['lines'])}):")
for cl in b['lines']:
    fields = cl.fields
    form   = fields[-1]
    tag    = fields[-2] if len(fields) >= 2 else ''
    print(f"    form={form!r:10s}  tag={tag!r}")


---
## 3. Finding the `.MKTARGETNEW` Target Word

After the MK block ends, the next phonemic corpus line in the same sentence is the
"target" word that the makura-kotoba modifies.  The script scans forward from the last
sentinel line to find this word and its lemma ID (if already annotated).

The `.MKTARGETNEW` field stores `ref_target=<ID>  <form>` (similar to `.COMPOUND`).
When `RELATED_TOP_N > 1`, the N most frequent targets across all corpora are stored.

In [ ]:
LEMMA_RE = re.compile(r'^[A-Za-z]\d+[a-z]*$')
PHON_TAGS = frozenset({
    "PHON", "LOG", "PHON-KUN", "PHON-ON",
    "NLOG", "PLOG", "BPHON", "ILL", "ORDLOG", "NLPOG",
})

# Pre-load all docs
_docs_cache = {}
for fname in sorted(os.listdir(TEXT_DIR)):
    if fname.endswith('.xml'):
        _docs_cache[fname] = CorpusDocument.from_file(os.path.join(TEXT_DIR, fname))

def find_next_phonemic_after(doc, last_cl):
    """Return (lemma_id_or_None, form) of first phonemic line after last_cl in doc."""
    found = False
    for cl in doc.all_corpus_lines():
        if not found:
            if cl is last_cl:
                found = True
            continue
        fields = cl.fields
        if len(fields) < 2:
            continue
        tag_base = re.sub(r';@\d+$', '', fields[-2])
        if tag_base in PHON_TAGS:
            form  = fields[-1]
            lemma = fields[-3] if len(fields) >= 3 and LEMMA_RE.match(fields[-3]) else None
            return lemma, form
    return None, None

# Find targets for first 5 blocks
print("Block → target word:")
for b in blocks[:5]:
    doc = _docs_cache.get(b['file'])
    if doc is None:
        continue
    last_cl = b['lines'][-1]
    tgt_lemma, tgt_form = find_next_phonemic_after(doc, last_cl)
    block_forms = [cl.fields[-1] for cl in b['lines']]
    print(f"  MK='{' '.join(block_forms)}'  →  target: {tgt_form!r}  (lemma: {tgt_lemma})")


---
## 4. Existing MK Entries in the Dictionary

Entries with `.POS makura kotoba` already in the dictionary may need their
`.COMPOUND` and `.MKTARGETNEW` fields filled in.  `NORMALISE_EXISTING = True`
triggers a second pass that scans the corpus to backfill these fields.

In [ ]:
from coj.core.dictionary import Dictionary

d = Dictionary.from_file(DICT_FILE)

mk_entries = [e for e in d.values() if 'makura kotoba' in (e.get_first('.POS') or '')]
print(f"Existing MK entries: {len(mk_entries)}")

# Check which are missing .COMPOUND or .MKTARGETNEW
missing_compound     = [e for e in mk_entries if not e.get_all('.COMPOUND')]
missing_mktargetnew  = [e for e in mk_entries if not e.get_all('.MKTARGETNEW')]

print(f"  Missing .COMPOUND    : {len(missing_compound)}")
print(f"  Missing .MKTARGETNEW : {len(missing_mktargetnew)}")

print("\nSample MK entry:")
if mk_entries:
    e = mk_entries[0]
    print(f"  ID     : {e.eid}")
    print(f"  GLOSS  : {e.get_first('.GLOSS')}")
    print(f"  FORMs  : {e.get_all('.FORM')}")
    print(f"  POS    : {e.get_first('.POS')}")
    print(f"  COMPOUND   : {e.get_all('.COMPOUND')}")
    print(f"  MKTARGETNEW: {e.get_all('.MKTARGETNEW')}")

---
## 5. Building a New MK Dictionary Entry

When a new MK block is encountered, the script calls `build_mk_entry()` to construct
a new dictionary block.  The entry concatenates:
- `.FORM` = component forms joined by space
- `.KANA` = katakana conversion of each form, space-joined
- `.GLOSS` = space-joined forms (same as form)
- `.MEANING` = `[makura-kotoba]`
- `.POS` = `makura kotoba`
- `.COMPOUND ref_target=<comp_id>  <form>` for each component
- `.MKTARGETNEW ref_target=<target_id>  <target_form>` for the following word

In [ ]:
from coj.core.kana import phonemic_to_kana

def build_mk_entry_preview(
    new_id: str,
    component_ids: list,
    component_forms: list,
    target_id,
    target_form,
) -> str:
    """Return a preview of the dictionary block text."""
    combined_form = ' '.join(component_forms)
    combined_kana = ' '.join(phonemic_to_kana(f) for f in component_forms)
    lines = [
        f"=== {new_id}",
        f".GLOSS\t{combined_form}",
        ".MEANING\t[makura-kotoba]",
        f".FORM\t{combined_form}",
        f".KANA\t{combined_kana}",
        ".POS\tmakura kotoba",
    ]
    for cid, cform in zip(component_ids, component_forms):
        lines.append(f".COMPOUND\tref_target={cid}  {cform}")
    if target_form:
        tid = target_id or 'UNKNOWN'
        lines.append(f".MKTARGETNEW\tref_target={tid}  {target_form}")
    return '\n'.join(lines)

# Simulate for block 0
b0 = blocks[0]
comp_forms = [cl.fields[-1] for cl in b0['lines']]
comp_ids   = ['L000001', 'L000002'][:len(comp_forms)]  # placeholder IDs

doc0 = _docs_cache.get(b0['file'])
last_cl = b0['lines'][-1]
tgt_lemma, tgt_form = find_next_phonemic_after(doc0, last_cl)

preview = build_mk_entry_preview(
    new_id='L099001',
    component_ids=comp_ids,
    component_forms=comp_forms,
    target_id=tgt_lemma,
    target_form=tgt_form,
)
print("Simulated new dictionary entry:")
print(preview)


---
## 6. The `.MKTARGETNEW` Field

`.MKTARGETNEW` is a multi-valued field (new in v2.0.0, replacing `.RELATED` for MK
entries) registered in `src/oncoj/tags.py`.

When `RELATED_TOP_N = N`, up to N target words are stored per entry, ordered by
frequency across all corpus files.

In [ ]:
from coj.core.tags import MULTI_VALUE_FIELDS

print(".MKTARGETNEW in MULTI_VALUE_FIELDS:", '.MKTARGETNEW' in MULTI_VALUE_FIELDS)

# Show any existing entries that already have .MKTARGETNEW
has_target = [e for e in mk_entries if e.get_all('.MKTARGETNEW')]
print(f"\nMK entries with .MKTARGETNEW: {len(has_target)}")
for e in has_target[:3]:
    print(f"  {e.eid}  form={e.get_first('.FORM')!r}")
    for t in e.get_all('.MKTARGETNEW'):
        print(f"    .MKTARGETNEW  {t}")

---
## 7. Running the Script

Configure the `USER SETTINGS` block in `scripts/mk_lemma_processor.py`, then:

```bash
python3 scripts/mk_lemma_processor.py
```

Key settings:

| Setting | Default | Effect |
|---|---|---|
| `OVERWRITE_SOURCE` | `False` | Write `*_processed.txt` to `OUTPUT_FOLDER` instead of editing source |
| `CREATE_DICT_ENTRIES` | `True` | Create new dictionary entries for each new MK block |
| `RELATED_TOP_N` | `1` | Number of `.MKTARGETNEW` target words to record per entry |
| `NORMALISE_EXISTING` | `True` | Fill missing `.COMPOUND`/`.MKTARGETNEW` in existing MK entries |
| `LEMMA_PREFIX` | `"L"` | Prefix for newly generated MK IDs |
| `TARGET_ID` | `"L099999"` | The sentinel ID to replace |